# Multi-Class Waste Detection Training Pipeline (YOLOv12)
Notebook ini digunakan untuk melatih model object detection **YOLOv12** pada dataset sampah jalur pendakian gunung dari Roboflow.

### 1. Periksa GPU & Instalasi Library

In [ ]:
!pip install ultralytics roboflow opencv-python pyyaml matplotlib

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

### 2. Download Dataset dari Roboflow (YOLOv12 Format)

In [ ]:
import os
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("weapon-mpr3p").project("ta-rhzr5")
version = project.version(1)
dataset = version.download("yolov12")
print("Dataset downloaded to:", dataset.location)

### 3. Normalisasi Path di data.yaml

In [ ]:
import yaml
from pathlib import Path

yaml_path = Path(dataset.location) / "data.yaml"
with open(yaml_path, "r") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(Path(dataset.location).resolve()).replace("\\", "/")
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
cfg["test"] = "test/images"

with open(yaml_path, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Kelas objek:", cfg.get("names"))
print("data.yaml berhasil disesuaikan!")

### 4. Unduh Pre-trained Weights YOLOv12 & Mulai Training

In [ ]:
import os, urllib.request
from ultralytics import YOLO

# Pilih model: yolov12n.pt (cepat/ringan) atau yolov12s.pt (seimbang)
model_name = "yolov12s.pt"
weights_url = f"https://github.com/sunsmarterjie/yolov12/releases/download/v1.0/{model_name}"

if not os.path.exists(model_name):
    print(f"Mengunduh {model_name}...")
    urllib.request.urlretrieve(weights_url, model_name)
    print("Download selesai.")

# Inisialisasi model
model = YOLO(model_name)

# Training
results = model.train(
    data=str(yaml_path),
    epochs=100,
    batch=16,
    imgsz=640,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=4,
    patience=30,
    project="runs/detect",
    name="waste_yolov12",
    plots=True
)

### 5. Evaluasi Model pada Data Test

In [ ]:
best_weights = "runs/detect/waste_yolov12/weights/best.pt"
best_model = YOLO(best_weights)
test_metrics = best_model.val(data=str(yaml_path), split="test")

print(f"mAP@50     : {test_metrics.box.map50:.4f}")
print(f"mAP@50-95  : {test_metrics.box.map:.4f}")
print(f"Precision  : {test_metrics.box.mp:.4f}")
print(f"Recall     : {test_metrics.box.mr:.4f}")

### 6. Uji Coba Prediksi / Visualisasi Deteksi

In [ ]:
import glob
import matplotlib.pyplot as plt
import cv2

# Ambil contoh gambar dari folder test
test_images = glob.glob(f"{dataset.location}/test/images/*.jpg")
sample_image = test_images[0] if test_images else None

if sample_image:
    pred_results = best_model.predict(source=sample_image, conf=0.25, save=False)
    annotated_img = pred_results[0].plot()
    
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("Hasil Deteksi Sampah")
    plt.show()